In [1]:
"""
FILE 2: model_training.py
==========================
Loads feature-engineered CSVs produced by feature_engineering.py
and trains three models:
  Model 1 → High-Value Customer Prediction
  Model 2 → Churn Risk Prediction
  Model 3 → Fraud Detection

IMPORTANT: Always run feature_engineering.py FIRST.
"""

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2E86AB', '#27AE60', '#E74C3C']

In [2]:
# ════════════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════════════

def evaluate_model(name, model, Xtr, Xte, ytr, yte, label=""):
    model.fit(Xtr, ytr)
    preds = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    acc   = accuracy_score(yte, preds)
    auc   = roc_auc_score(yte, proba)
    cv    = cross_val_score(model, Xtr, ytr, cv=5, scoring='roc_auc').mean()
    print(f"  [{name}]  Acc={acc:.4f}  AUC={auc:.4f}  CV-AUC={cv:.4f}")
    print(classification_report(yte, preds, target_names=[f'Not {label}', label]))
    return model, preds, proba, auc


def plot_roc(ax, results, title):
    for name, yte, proba, color in results:
        fpr, tpr, _ = roc_curve(yte, proba)
        auc = roc_auc_score(yte, proba)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} AUC={auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set(xlabel='FPR', ylabel='TPR', title=title)
    ax.legend(loc='lower right', fontsize=8)


def plot_importance(ax, model, features, title, color, top_n=12):
    if hasattr(model, 'feature_importances_'):
        imp = pd.Series(model.feature_importances_, index=features)
    else:
        imp = pd.Series(np.abs(model.coef_[0]), index=features)
    imp.nlargest(top_n).sort_values().plot(kind='barh', ax=ax, color=color)
    ax.set(title=title, xlabel='Importance')


def pick_best(candidates):
    return max(candidates, key=lambda x: x[0])

In [3]:
# ════════════════════════════════════════════════════════════════
# STEP 1: LOAD FEATURE-ENGINEERED DATA
# ════════════════════════════════════════════════════════════════
print("=" * 60)
print("LOADING FEATURE-ENGINEERED DATA")
print("=" * 60)

try:
    cust_eng   = pd.read_csv("customer_engineered.csv")
    claims_eng = pd.read_csv("claims_engineered.csv")
    with open("engineering_metadata.pkl", "rb") as f:
        meta = pickle.load(f)
    print(f"customer_engineered.csv : {cust_eng.shape}")
    print(f"claims_engineered.csv   : {claims_eng.shape}")
    print(f"Metadata loaded         : premium_75th = {meta['premium_75th']:.2f}")
except FileNotFoundError as e:
    print(f"\nERROR: {e}")
    print("Please run  python feature_engineering.py  first!")
    exit(1)

LOADING FEATURE-ENGINEERED DATA
customer_engineered.csv : (510, 29)
claims_engineered.csv   : (1000, 78)
Metadata loaded         : premium_75th = 378.68


In [4]:
# ════════════════════════════════════════════════════════════════
# STEP 2: PREPARE CUSTOMER FEATURES
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PREPARING CUSTOMER FEATURES")
print("=" * 60)

# Columns to exclude from features
EXCLUDE_CUST = {'Customer_ID', 'Policy_Type', 'is_high_value', 'churn_risk'}
FEAT_CUST = [c for c in cust_eng.columns
             if c not in EXCLUDE_CUST
             and cust_eng[c].dtype in [np.float64, np.int64, bool, np.uint8]]

print(f"Customer feature columns ({len(FEAT_CUST)}): {FEAT_CUST}")


PREPARING CUSTOMER FEATURES
Customer feature columns (26): ['Age', 'Premium', 'Claim_Count', 'Customer_Satisfaction', 'tenure_years', 'claims_per_year', 'age_x_claims', 'premium_x_satisfaction', 'satisfaction_per_tenure', 'Gender_Female', 'Gender_Male', 'Gender_Other', 'Gender_Unknown', 'Region_East', 'Region_North', 'Region_South', 'Region_West', 'age_group_Young', 'age_group_Mid', 'age_group_Senior', 'premium_tier_Low', 'premium_tier_Mid', 'premium_tier_High', 'satisfaction_group_Low', 'satisfaction_group_Mid', 'satisfaction_group_High']


In [5]:
# ════════════════════════════════════════════════════════════════
# STEP 3: CREATE TARGET VARIABLES
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("CREATING TARGET VARIABLES")
print("=" * 60)

# Target 1: High-Value Customer (top 25% premium)
threshold = meta['premium_75th']
cust_eng['is_high_value'] = (cust_eng['Premium'] > threshold).astype(int)
print(f"High-value threshold (75th pct): Rs{threshold:.2f}")
print(f"High-value customers           : {cust_eng['is_high_value'].sum()} / {len(cust_eng)}")

# Target 2: Churn Risk
# (low satisfaction OR high claims rate with short tenure)
churn_mask = (
    (cust_eng['Customer_Satisfaction'] < 4) |
    ((cust_eng['claims_per_year'] > cust_eng['Claim_Count'].median()) &
     (cust_eng['tenure_years'] < 2.0))
)
cust_eng['churn_risk'] = churn_mask.astype(int)
print(f"\nChurn-risk customers           : {cust_eng['churn_risk'].sum()} "
      f"({cust_eng['churn_risk'].mean()*100:.1f}%)")

# Target 3: Fraud (already in claims data)
print(f"\nFraud rate in claims           : {(claims_eng['is_fraud'].mean()*100):.1f}%")


CREATING TARGET VARIABLES
High-value threshold (75th pct): Rs378.68
High-value customers           : 128 / 510

Churn-risk customers           : 183 (35.9%)

Fraud rate in claims           : 24.7%


In [6]:
# ════════════════════════════════════════════════════════════════
# MODEL 1: HIGH-VALUE CUSTOMER PREDICTION
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("MODEL 1: HIGH-VALUE CUSTOMER PREDICTION")
print("=" * 60)

X1 = cust_eng[FEAT_CUST].fillna(0)
y1 = cust_eng['is_high_value']

X1_tr, X1_te, y1_tr, y1_te = train_test_split(
    X1, y1, test_size=0.2, random_state=SEED, stratify=y1)

sc1 = StandardScaler()
X1_tr_s = sc1.fit_transform(X1_tr)
X1_te_s = sc1.transform(X1_te)

lr1, lr1_p, lr1_pr, lr1_auc = evaluate_model("Logistic Regression",
    LogisticRegression(max_iter=1000, random_state=SEED),
    X1_tr_s, X1_te_s, y1_tr, y1_te, "High-Value")

rf1, rf1_p, rf1_pr, rf1_auc = evaluate_model("Random Forest",
    RandomForestClassifier(n_estimators=200, max_depth=8, random_state=SEED),
    X1_tr_s, X1_te_s, y1_tr, y1_te, "High-Value")

gb1, gb1_p, gb1_pr, gb1_auc = evaluate_model("Gradient Boosting",
    GradientBoostingClassifier(n_estimators=200, max_depth=5,
                               learning_rate=0.05, random_state=SEED),
    X1_tr_s, X1_te_s, y1_tr, y1_te, "High-Value")

best1_auc, best1_mdl, best1_lbl, best1_pr, best1_p = pick_best([
    (lr1_auc, lr1, "Logistic Reg",      lr1_pr, lr1_p),
    (rf1_auc, rf1, "Random Forest",     rf1_pr, rf1_p),
    (gb1_auc, gb1, "Gradient Boosting", gb1_pr, gb1_p)])
print(f"\n  Best Model 1: {best1_lbl}  AUC={best1_auc:.4f}")


MODEL 1: HIGH-VALUE CUSTOMER PREDICTION
  [Logistic Regression]  Acc=0.9804  AUC=0.9975  CV-AUC=0.9915
                precision    recall  f1-score   support

Not High-Value       0.99      0.99      0.99        76
    High-Value       0.96      0.96      0.96        26

      accuracy                           0.98       102
     macro avg       0.97      0.97      0.97       102
  weighted avg       0.98      0.98      0.98       102

  [Random Forest]  Acc=1.0000  AUC=1.0000  CV-AUC=1.0000
                precision    recall  f1-score   support

Not High-Value       1.00      1.00      1.00        76
    High-Value       1.00      1.00      1.00        26

      accuracy                           1.00       102
     macro avg       1.00      1.00      1.00       102
  weighted avg       1.00      1.00      1.00       102

  [Gradient Boosting]  Acc=1.0000  AUC=1.0000  CV-AUC=0.9952
                precision    recall  f1-score   support

Not High-Value       1.00      1.00      1.

In [7]:
# ════════════════════════════════════════════════════════════════
# MODEL 2: CHURN RISK PREDICTION
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("MODEL 2: CHURN RISK PREDICTION")
print("=" * 60)

X2 = cust_eng[FEAT_CUST].fillna(0)
y2 = cust_eng['churn_risk']

X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.2, random_state=SEED, stratify=y2)

sc2 = StandardScaler()
X2_tr_s = sc2.fit_transform(X2_tr)
X2_te_s = sc2.transform(X2_te)

lr2, lr2_p, lr2_pr, lr2_auc = evaluate_model("Logistic Regression",
    LogisticRegression(max_iter=1000, random_state=SEED),
    X2_tr_s, X2_te_s, y2_tr, y2_te, "Churn")

rf2, rf2_p, rf2_pr, rf2_auc = evaluate_model("Random Forest",
    RandomForestClassifier(n_estimators=200, max_depth=8,
                           class_weight='balanced', random_state=SEED),
    X2_tr_s, X2_te_s, y2_tr, y2_te, "Churn")

gb2, gb2_p, gb2_pr, gb2_auc = evaluate_model("Gradient Boosting",
    GradientBoostingClassifier(n_estimators=200, max_depth=5,
                               learning_rate=0.05, random_state=SEED),
    X2_tr_s, X2_te_s, y2_tr, y2_te, "Churn")

best2_auc, best2_mdl, best2_lbl, best2_pr, best2_p = pick_best([
    (lr2_auc, lr2, "Logistic Reg",      lr2_pr, lr2_p),
    (rf2_auc, rf2, "Random Forest",     rf2_pr, rf2_p),
    (gb2_auc, gb2, "Gradient Boosting", gb2_pr, gb2_p)])
print(f"\n  Best Model 2: {best2_lbl}  AUC={best2_auc:.4f}")


MODEL 2: CHURN RISK PREDICTION
  [Logistic Regression]  Acc=0.9706  AUC=1.0000  CV-AUC=0.9931
              precision    recall  f1-score   support

   Not Churn       0.96      1.00      0.98        65
       Churn       1.00      0.92      0.96        37

    accuracy                           0.97       102
   macro avg       0.98      0.96      0.97       102
weighted avg       0.97      0.97      0.97       102

  [Random Forest]  Acc=1.0000  AUC=1.0000  CV-AUC=1.0000
              precision    recall  f1-score   support

   Not Churn       1.00      1.00      1.00        65
       Churn       1.00      1.00      1.00        37

    accuracy                           1.00       102
   macro avg       1.00      1.00      1.00       102
weighted avg       1.00      1.00      1.00       102

  [Gradient Boosting]  Acc=1.0000  AUC=1.0000  CV-AUC=1.0000
              precision    recall  f1-score   support

   Not Churn       1.00      1.00      1.00        65
       Churn       1.00 

In [8]:
# ════════════════════════════════════════════════════════════════
# MODEL 3: FRAUD DETECTION
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("MODEL 3: FRAUD DETECTION")
print("=" * 60)

EXCLUDE_FR = {'is_fraud', 'fraud_reported', 'policy_number',
              'insured_zip', '_c39'}
FEAT_FR = [c for c in claims_eng.columns
           if c not in EXCLUDE_FR
           and claims_eng[c].dtype in [np.float64, np.int64, np.uint8, bool]]

X3 = claims_eng[FEAT_FR].fillna(claims_eng[FEAT_FR].median())
y3 = claims_eng['is_fraud']

print(f"Fraud features: {len(FEAT_FR)}")

X3_tr, X3_te, y3_tr, y3_te = train_test_split(
    X3, y3, test_size=0.2, random_state=SEED, stratify=y3)

sc3 = StandardScaler()
X3_tr_s = sc3.fit_transform(X3_tr)
X3_te_s = sc3.transform(X3_te)

lr3, lr3_p, lr3_pr, lr3_auc = evaluate_model("Logistic Regression",
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED),
    X3_tr_s, X3_te_s, y3_tr, y3_te, "Fraud")

rf3, rf3_p, rf3_pr, rf3_auc = evaluate_model("Random Forest",
    RandomForestClassifier(n_estimators=200, max_depth=8,
                           class_weight='balanced', random_state=SEED),
    X3_tr_s, X3_te_s, y3_tr, y3_te, "Fraud")

gb3, gb3_p, gb3_pr, gb3_auc = evaluate_model("Gradient Boosting",
    GradientBoostingClassifier(n_estimators=200, max_depth=5,
                               learning_rate=0.05, random_state=SEED),
    X3_tr_s, X3_te_s, y3_tr, y3_te, "Fraud")

best3_auc, best3_mdl, best3_lbl, best3_pr, best3_p = pick_best([
    (lr3_auc, lr3, "Logistic Reg",      lr3_pr, lr3_p),
    (rf3_auc, rf3, "Random Forest",     rf3_pr, rf3_p),
    (gb3_auc, gb3, "Gradient Boosting", gb3_pr, gb3_p)])
print(f"\n  Best Model 3: {best3_lbl}  AUC={best3_auc:.4f}")


MODEL 3: FRAUD DETECTION
Fraud features: 66
  [Logistic Regression]  Acc=0.8350  AUC=0.8100  CV-AUC=0.7344
              precision    recall  f1-score   support

   Not Fraud       0.92      0.86      0.89       151
       Fraud       0.64      0.76      0.69        49

    accuracy                           0.83       200
   macro avg       0.78      0.81      0.79       200
weighted avg       0.85      0.83      0.84       200

  [Random Forest]  Acc=0.8200  AUC=0.7786  CV-AUC=0.7529
              precision    recall  f1-score   support

   Not Fraud       0.88      0.88      0.88       151
       Fraud       0.63      0.63      0.63        49

    accuracy                           0.82       200
   macro avg       0.76      0.76      0.76       200
weighted avg       0.82      0.82      0.82       200

  [Gradient Boosting]  Acc=0.7700  AUC=0.7542  CV-AUC=0.7351
              precision    recall  f1-score   support

   Not Fraud       0.82      0.89      0.85       151
       Frau

In [9]:
# ════════════════════════════════════════════════════════════════
# DASHBOARD
# ════════════════════════════════════════════════════════════════
print("\nGenerating dashboard...")
fig = plt.figure(figsize=(20, 18))
fig.suptitle('Insurance Analytics — Three Model Dashboard',
             fontsize=18, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

plot_roc(fig.add_subplot(gs[0, 0]),
         [("Log Reg", y1_te, lr1_pr, COLORS[0]),
          ("Rand For", y1_te, rf1_pr, COLORS[1]),
          ("Grad Bst", y1_te, gb1_pr, COLORS[2])],
         "ROC — High-Value Customer")
plot_roc(fig.add_subplot(gs[1, 0]),
         [("Log Reg", y2_te, lr2_pr, COLORS[0]),
          ("Rand For", y2_te, rf2_pr, COLORS[1]),
          ("Grad Bst", y2_te, gb2_pr, COLORS[2])],
         "ROC — Churn Risk")
plot_roc(fig.add_subplot(gs[2, 0]),
         [("Log Reg", y3_te, lr3_pr, COLORS[0]),
          ("Rand For", y3_te, rf3_pr, COLORS[1]),
          ("Grad Bst", y3_te, gb3_pr, COLORS[2])],
         "ROC — Fraud Detection")

plot_importance(fig.add_subplot(gs[0, 1]), best1_mdl, FEAT_CUST,
                f"Top Features — High-Value\n(Best: {best1_lbl})", COLORS[0])
plot_importance(fig.add_subplot(gs[1, 1]), best2_mdl, FEAT_CUST,
                f"Top Features — Churn\n(Best: {best2_lbl})", COLORS[1])
plot_importance(fig.add_subplot(gs[2, 1]), best3_mdl, FEAT_FR,
                f"Top Features — Fraud\n(Best: {best3_lbl})", COLORS[2])

for ax, yte, pred, title, cmap in [
    (fig.add_subplot(gs[0, 2]), y1_te, best1_p,
     f"Confusion — High-Value\nAUC={best1_auc:.3f}", "Blues"),
    (fig.add_subplot(gs[1, 2]), y2_te, best2_p,
     f"Confusion — Churn\nAUC={best2_auc:.3f}", "Greens"),
    (fig.add_subplot(gs[2, 2]), y3_te, best3_p,
     f"Confusion — Fraud\nAUC={best3_auc:.3f}", "Reds"),
]:
    cm = confusion_matrix(yte, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                cbar=False, annot_kws={'size': 13})
    ax.set(title=title, xlabel='Predicted', ylabel='Actual')

plt.savefig('model_dashboard.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: model_dashboard.png")


Generating dashboard...
  Saved: model_dashboard.png


In [10]:
# ════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
summary = pd.DataFrame({
    'Model':         ['High-Value Customer', 'Churn Risk', 'Fraud Detection'],
    'Best Algorithm':[best1_lbl, best2_lbl, best3_lbl],
    'Test AUC':      [round(best1_auc, 4), round(best2_auc, 4), round(best3_auc, 4)],
    'Accuracy':      [round(accuracy_score(y1_te, best1_p), 4),
                      round(accuracy_score(y2_te, best2_p), 4),
                      round(accuracy_score(y3_te, best3_p), 4)],
})
print(summary.to_string(index=False))
summary.to_csv("model_summary.csv", index=False)
print("\nSaved: model_summary.csv")


FINAL SUMMARY
              Model Best Algorithm  Test AUC  Accuracy
High-Value Customer  Random Forest      1.00    1.0000
         Churn Risk   Logistic Reg      1.00    0.9706
    Fraud Detection   Logistic Reg      0.81    0.8350

Saved: model_summary.csv


In [11]:
# ════════════════════════════════════════════════════════════════
# PREDICTION FUNCTIONS (load and use anywhere)
# ════════════════════════════════════════════════════════════════

def load_models(path="trained_models.pkl"):
    """Load trained models from disk."""
    with open(path, "rb") as f:
        return pickle.load(f)


def predict_high_value(customer_row: dict, bundle: dict) -> dict:
    """
    Predict if a customer is high-value.
    customer_row: dict with ENGINEERED feature values (run through feature_engineering first)
    """
    inp  = pd.DataFrame([customer_row]).reindex(columns=bundle['feat_cust'], fill_value=0)
    prob = bundle['model1_high_value'].predict_proba(bundle['scaler1'].transform(inp))[0][1]
    return {
        'high_value': bool(prob >= 0.5),
        'probability': round(float(prob), 4),
        'premium_threshold': round(bundle['meta']['premium_75th'], 2)
    }


def predict_churn(customer_row: dict, bundle: dict) -> dict:
    """
    Predict churn risk level for a customer.
    customer_row: dict with ENGINEERED feature values
    """
    inp  = pd.DataFrame([customer_row]).reindex(columns=bundle['feat_cust'], fill_value=0)
    prob = bundle['model2_churn'].predict_proba(bundle['scaler2'].transform(inp))[0][1]
    risk = 'High' if prob >= 0.6 else 'Medium' if prob >= 0.4 else 'Low'
    return {'churn_risk': risk, 'probability': round(float(prob), 4)}


def predict_fraud(claim_row: dict, bundle: dict) -> dict:
    """
    Predict if a claim is fraudulent.
    claim_row: dict with ENGINEERED claim feature values
    """
    inp   = pd.DataFrame([claim_row]).reindex(columns=bundle['feat_fr'], fill_value=0)
    prob  = bundle['model3_fraud'].predict_proba(bundle['scaler3'].transform(inp))[0][1]
    alert = 'HIGH' if prob >= 0.7 else 'MEDIUM' if prob >= 0.4 else 'LOW'
    return {
        'fraud_likely': bool(prob >= 0.5),
        'probability':  round(float(prob), 4),
        'alert_level':  alert
    }